## Step 1: Data Preprocessing


In [6]:
import pandas as pd
import json
import os
import spacy
spacy.load('en_core_web_sm')


# Path to the Excel file
# load the Excel file from the specified path
file_path = "/home/rooein/baby_lm_project/babylm-interaction/data/complexity_metrics/data/babyLM_Llama-3.2-3B_dialogue_starters_2025-08-08.xlsx"

# Read the Excel file
xls = pd.ExcelFile(file_path)

# Create an output directory to store the JSON files
output_dir = "/home/rooein/baby_lm_project/babylm-interaction/data/complexity_metrics/data/text"
os.makedirs(output_dir, exist_ok=True)

# Iterate over each sheet in the Excel file
for sheet_name in xls.sheet_names:
    # Read the sheet into a DataFrame
    df = pd.read_excel(xls, sheet_name=sheet_name)
    
    # Convert the DataFrame to a list of dictionaries (records)
    sheet_data = df.to_dict(orient='records')
    
    # Add an 'id' and change the name 'output' to 'text' in each entry
    for counter, entry in enumerate(sheet_data, start=1):
        entry['id'] = f"{counter}"
        entry['level']=f"{sheet_name}"
        # Rename 'output' key to 'text' if it exists
        if 'output' in entry:
            entry['text'] = entry.pop('output')

    # Create a JSON file for each sheet
    json_file_path = os.path.join(output_dir, f"{sheet_name}.json")
    with open(json_file_path, 'w') as json_file:
        json.dump(sheet_data, json_file, indent=4)

print("Sheets have been saved as JSON files with 'text' key instead of 'output'.")


Sheets have been saved as JSON files with 'text' key instead of 'output'.


## Step 2: Metric Calculation

In [4]:
import pandas as pd
import spacy
import nltk
import json
from textstat import flesch_kincaid_grade, coleman_liau_index, smog_index, gunning_fog, syllable_count
import taaled
import cefrpy
import requests
import os
from connectives_list import CONNECTIVES
import spacy
from compleximeter import ComplexiMeter
spacy.load('en_core_web_sm')


# Assuming the ComplexiMeter class is defined as in your original code

def process_json_data(input_json_file, output_csv_path):
    """
    Process a JSON file containing texts, compute complexity metrics for each text, and save the results to a CSV.
    
    Args:
    input_json_file (str): Path to the input JSON file.
    output_csv_path (str): Path to the output CSV file where results will be saved.
    """
    # Read the JSON file
    with open(input_json_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    # Initialize the ComplexiMeter
    meter = ComplexiMeter(metrics_file='Metrics.csv', crat_path='CRAT_v1.1.app')
    
    # Process the dataset and compute the metrics
    meter.process_dataset(data, output_csv_path)

    print(f"Results saved to {output_csv_path}")

if __name__ == '__main__':
    # add your input JSON file path and output CSV path here
    input_json_file = '/home/rooein/baby_lm_project/babylm-interaction/data/complexity_metrics/data/text/2-3years.json'  # Path to your input JSON file
    output_csv_path = '/home/rooein/baby_lm_project/babylm-interaction/data/complexity_metrics/results/complexity_results_2-3years.csv'  # Path to save the output CSV

    process_json_data(input_json_file, output_csv_path)


  - corpora/wordnet
  - corpora/omw-1.4
  - corpora/wordnet_ic
  - corpora/brown
  - taggers/averaged_perceptron_tagger_eng
Metrics file 'Metrics.csv' not found.
Starting to process 100 texts...
Successfully saved results for 100 texts to '/home/rooein/baby_lm_project/babylm-interaction/data/complexity_metrics/results/complexity_results_2-3years.csv'.
Results saved to /home/rooein/baby_lm_project/babylm-interaction/data/complexity_metrics/results/complexity_results_2-3years.csv


## Step 3: BabyLM Metric Filtering

In [8]:
# read the the output csv file and filter the metrics baed on a list
output_csv_path = '/home/rooein/baby_lm_project/babylm-interaction/data/complexity_metrics/results/complexity_results_2-3years.csv'  # Path to your output CSV file
results = pd.read_csv(output_csv_path)
selected_metrics = [
    'average_sentence_length',
    'pronouns_density',
    'first_person_pronouns_density',
    'third_person_pronouns_density',
    'verbs_density',
    'adverbs_density',
    'type_token_ratio',
    'mattr',
    'content_word_overlap_adjacent',
    'noun_overlap_adjacent',
    'argument_overlap_adjacent',
    'stem_overlap_sent',
    'percentage_of_words_above_b1_level',
    'average_cefr_level',
    'average_concreteness',
    'word_concreteness_cox',
    'referential_cohesion_cox',
    'deep_causal_cohesion_cox'
]

filtered_results = results[selected_metrics]

In [10]:
results.columns

Index(['id', 'input', 'academic_word_list_coverage', 'additive_connectives',
       'adjectives_density', 'adverbs_density', 'adversative_connectives',
       'argument_overlap_adjacent', 'argument_overlap_all',
       'average_age_of_acquisition',
       ...
       'verb_tense_repetition', 'verb_tense_repetition_nltk', 'verb_ttr',
       'verbs_density', 'word_concreteness_cox', 'word_frequency_log',
       'narrativity_score', 'referential_cohesion_score',
       'word_concreteness_score', 'deep_causal_cohesion_score'],
      dtype='object', length=110)

In [9]:
filtered_results

,average_sentence_length,pronouns_density,first_person_pronouns_density,third_person_pronouns_density,verbs_density,adverbs_density,type_token_ratio,mattr,content_word_overlap_adjacent,noun_overlap_adjacent,argument_overlap_adjacent,stem_overlap_sent,percentage_of_words_above_b1_level,average_cefr_level,average_concreteness,word_concreteness_cox,referential_cohesion_cox,deep_causal_cohesion_cox
0,9.0,0.250000,0.100000,0.050000,0.200000,0.000000,0.851852,0.851852,0.166667,0.333333,0.285714,0.083333,10.000000,1.650000,3.332727,1.110909,-0.275132,1.020833
1,17.0,0.142857,0.071429,0.000000,0.142857,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.071429,3.842857,1.280952,-0.300000,2.273810
2,12.0,0.187500,0.062500,0.000000,0.187500,0.000000,0.875000,0.875000,0.000000,0.000000,0.200000,0.066667,6.250000,1.382500,3.512222,1.170741,-0.355000,1.645833
3,17.0,0.230769,0.000000,0.153846,0.076923,0.038462,0.823529,0.823529,0.222222,0.200000,0.250000,0.136364,11.538462,1.575385,3.973333,1.324444,-0.275261,1.526515
4,8.5,0.166667,0.083333,0.083333,0.083333,0.166667,0.941176,0.941176,0.000000,0.000000,0.000000,0.000000,0.000000,1.208333,3.746667,1.248889,-0.288235,1.541667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,8.5,0.166667,0.000000,0.083333,0.083333,0.000000,0.941176,0.941176,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,3.536667,1.178889,-0.388235,1.968750
96,19.0,0.285714,0.142857,0.071429,0.142857,0.000000,0.842105,0.842105,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,3.488571,1.162857,-0.268421,1.791667
97,10.0,0.071429,0.071429,0.000000,0.214286,0.000000,1.000000,1.000000,0.500000,1.000000,0.666667,0.166667,0.000000,1.107143,4.041250,1.347083,-0.116667,1.031250
98,25.0,0.250000,0.150000,0.000000,0.200000,0.000000,0.880000,0.880000,0.000000,0.000000,0.000000,0.000000,0.000000,1.050000,4.136250,1.378750,-0.276000,0.833333


In [ ]:
# save the filtered results to a new CSV file